# Error generation using dspy

In [ ]:
# python data_generation/error_generation_dspy.py --csv_path ./csv --save_path ./csv

In [ ]:
# !pip install dspy pandas

# Question-answer items

In [ ]:
import pandas as pd

none_sample_df     = pd.read_csv('./csv/dspy_none.csv')
effusion_sample_df = pd.read_csv('./csv/dspy_effusion.csv')
nodule_sample_df   = pd.read_csv('./csv/dspy_nodule.csv')

omission_effusion   = effusion_sample_df[effusion_sample_df['omission_effusion_classification'] == 1]
insertion_nodule    = effusion_sample_df[effusion_sample_df['insertion_nodule_classification'] == 1]
direction_effusion  = effusion_sample_df[effusion_sample_df['direction_effusion_classification'] == 1]
size_effusion       = effusion_sample_df[effusion_sample_df['size_classification'] == 1]
typo_effusion       = effusion_sample_df[effusion_sample_df['typo_classification'] == 1]
unit_effusion       = effusion_sample_df[effusion_sample_df['unit_classification'] == 1]

omission_nodule    = nodule_sample_df[nodule_sample_df['omission_nodule_classification'] == 1]
insertion_effusion = nodule_sample_df[nodule_sample_df['insertion_effusion_classification'] == 1]
direction_nodule   = nodule_sample_df[nodule_sample_df['direction_nodule_classification'] == 1]
size_nodule        = nodule_sample_df[nodule_sample_df['size_classification'] == 1]
typo_nodule        = nodule_sample_df[nodule_sample_df['typo_classification'] == 1]
unit_nodule        = nodule_sample_df[nodule_sample_df['unit_classification'] == 1]

none_insert_nodule   = none_sample_df[none_sample_df['insertion_nodule_classification'] == 1]
none_insert_effusion = none_sample_df[none_sample_df['insertion_effusion_classification'] == 1]

## Question format

In [ ]:
question_format = {
    "classification": [
        "Evaluate the accuracy of the provided medical report based on the given CT scan image. Respond strictly with either 0 (report accurate) or 1 (report contains inaccuracies).\nAnswer Format: '0' or '1'",
        "Check the provided medical report for correctness using the CT scan image as a reference. Your response must strictly be either 0 (report is correct) or 1 (report is incorrect).\nAnswer Format: '0' or '1'",
        "Determine if the medical report accurately describes the provided CT scan. Answer strictly 0 if the report is correct, or 1 if it contains any errors.\nAnswer Format: '0' or '1'",
        "Assess the medical report for any errors or inaccuracies using the provided CT scan image as the standard reference. Return exactly 0 if no errors exist, or exactly 1 if errors are present.\nAnswer Format: '0' or '1'",
        "Verify whether the medical report is accurate based on the provided CT scan. Return strictly 0 for correct or 1 if any inaccuracies exist.\nAnswer Format: '0' or '1'"
    ],    
    
    "detection": [
        "Identify and extract sentences from the provided medical report that contain inaccuracies or inconsistencies relative to the image. If none exist, reply: 'No errors detected.'\nAnswer Format: '{error sentence}' or 'No errors detected.'",
        "Detect report-image errors. Output erroneous sentence(s) or 'No errors detected.'\nAnswer Format: '{error sentence}' or 'No errors detected.'",
        "Check the medical report for any errors, including mismatches with the image. If any such issues are found, point out the exact sentence(s) involved. If everything is accurate and consistent, say: 'No errors detected.'\nAnswer Format: '{error sentence}' or 'No errors detected.'",
        "Scan the medical report for any errors or factual discrepancies in comparison to the image. Return erroneous sentence(s) or respond with 'No errors detected.'\nAnswer Format: '{error sentence}' or 'No errors detected.'",
        "Highlight any incorrect, mismatched, or erroneous sentences in the medical report, including those that conflict with the provided image. If no issues are found, respond clearly with 'No errors detected.'\nAnswer Format: '{error sentence}' or 'No errors detected.'"
    ],   
    
    "detection_index": [
        "Identify and extract the index numbers of sentences in the provided medical report that contain inaccuracies or inconsistencies with reference to the image. If no errors are found, output '0' to indicate no issues.\nAnswer Format: '{index}' or '0'",
        "Detect report-image errors. Output the index of each erroneous sentence. If there are no errors, respond with '0' to indicate all sentences are correct.\nAnswer Format: '{index}' or '0'",
        "Check the medical report for any errors, including mismatches with the image. If any such issues are found, return the index numbers of the sentences involved. If everything is consistent, output '0' to indicate no errors.\nAnswer Format: '{index}' or '0'",
        "Scan the medical report for any errors or factual discrepancies in comparison to the image. Return the index numbers of any erroneous sentences. If no such sentences exist, respond with '0' to indicate no issues were detected.\nAnswer Format: '{index}' or '0'",
        "Highlight any incorrect, mismatched, or erroneous sentences in the medical report, including those that conflict with the provided image. If the report is fully accurate, clearly respond with '0' to show that no errors were found.\nAnswer Format: '{index}' or '0'"
    ],    
    
    "correction": [
        "Review the medical report in comparison with the provided image. First, identify any sentence(s) that are inaccurate or any important findings from the image that are missing from the report. Then, provide a corrected version of the inaccurate sentence(s) or write the missing sentence(s) that should have been included. If no issues are found, respond: 'No errors detected.'\nAnswer Format: '{corrected or added sentence}' or 'No errors detected.'",
        "Compare the medical report with the corresponding image. First, identify any erroneous or omitted sentence(s) based on the image. Then, revise the incorrect sentence(s) or generate the sentence(s) that were missing. If the report is accurate and complete, respond: 'No errors detected.'\nAnswer Format: '{corrected or added sentence}' or 'No errors detected.'",
        "Analyze the medical report with reference to the provided image. First, detect any errors or missing descriptions. Then, return the corrected version of the inaccurate content or the missing sentence(s). If everything is accurate, respond with 'No errors detected.'\nAnswer Format: '{corrected or added sentence}' or 'No errors detected.'",
        "Evaluate the medical report with the given image. Begin by identifying any inaccurate or missing statement(s) in the report. After identifying them, present either the corrected version or the appropriate missing sentence(s). If no problems are found, respond: 'No errors detected.'\nAnswer Format: '{corrected or added sentence}' or 'No errors detected.'",
        "Examine the medical report alongside the associated image. First, identify any inaccuracies or omissions. Then, provide the corrected version of the sentence(s) or the sentence(s) that should be added. If the report is both accurate and complete, reply with: 'No errors detected.'\nAnswer Format: '{corrected or added sentence}' or 'No errors detected.'"
    ],
}

## level 1

### insertion

In [ ]:
import random
from tqdm import tqdm
import json

def generate_insert_conversations(df, lesion_type, report_col, inserted_sentence_col, question_format, start_index=0):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['classification'])
        report = sample[report_col]
        distorted_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{distorted_sentence}"

        human_dict = {
            'type': 'insertion',
            'level': '1',
            'lesion': lesion_type,
            'from': 'human',
            'value': input_text.strip()
        }

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# run
full_list = []
full_list += generate_insert_conversations(none_insert_nodule, 
                                           lesion_type='nodule',
                                           report_col='insertion_nodule_distorted_report',
                                           inserted_sentence_col='insertion_nodule_classification',
                                           question_format=question_format,
                                           start_index=0)

full_list += generate_insert_conversations(none_insert_effusion, 
                                           lesion_type='effusion',
                                           report_col='insertion_effusion_distorted_report',
                                           inserted_sentence_col='insertion_effusion_classification',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_insert_conversations(insertion_effusion, 
                                           lesion_type='effusion',
                                           report_col='insertion_effusion_distorted_report',
                                           inserted_sentence_col='insertion_effusion_classification',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_insert_conversations(insertion_nodule, 
                                           lesion_type='nodule',
                                           report_col='insertion_nodule_distorted_report',
                                           inserted_sentence_col='insertion_nodule_classification',
                                           question_format=question_format,
                                           start_index=len(full_list))


### omission

In [ ]:
import random
from tqdm import tqdm
import json

def generate_omit_conversations(df, lesion_type, report_col, inserted_sentence_col, question_format, start_index=0):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['classification'])
        report = sample[report_col]
        distorted_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{distorted_sentence}"

        human_dict = {
            'type': 'omission',
            'level': '1',
            'lesion': lesion_type,
            'from': 'human',
            'value': input_text.strip()
        }

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list



# run
full_list += generate_omit_conversations(omission_nodule, 
                                         lesion_type='nodule',
                                         report_col='omission_nodule_distorted_report',
                                         inserted_sentence_col='omission_nodule_classification',
                                         question_format=question_format,
                                         start_index=len(full_list))

full_list += generate_omit_conversations(omission_effusion, 
                                         lesion_type='effusion',
                                         report_col='omission_effusion_distorted_report',
                                         inserted_sentence_col='omission_effusion_classification',
                                         question_format=question_format,
                                         start_index=len(full_list))



### others

In [ ]:
import random
from tqdm import tqdm
import json

def generate_other_conversations(df, report_col, inserted_sentence_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['classification'])
        report = sample[report_col]
        distorted_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{distorted_sentence}"
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '1',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '1',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# nodule
full_list += generate_other_conversations(direction_nodule, 
                                          lesion_type='nodule',
                                          report_col='direction_nodule_distorted_report',
                                          inserted_sentence_col='direction_nodule_classification',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_nodule, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_classification',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_nodule, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_classification',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_nodule, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_classification',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))



# effusion
full_list += generate_other_conversations(direction_effusion, 
                                          lesion_type='effusion',
                                          report_col='direction_effusion_distorted_report',
                                          inserted_sentence_col='direction_effusion_classification',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_effusion, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_classification',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_effusion, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_classification',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_effusion, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_classification',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))


### No errors detected

In [ ]:
import random
from tqdm import tqdm
import json

def generate_normal_conversations(df, report_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['classification'])
        report = sample[report_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = "0"
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '1',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '1',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list



# normal
full_list += generate_normal_conversations(nodule_sample_df, 
                                           lesion_type='nodule',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_normal_conversations(effusion_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 

full_list += generate_normal_conversations(none_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 



In [ ]:
len(full_list)

## level 2

### insertion

In [ ]:
import random
from tqdm import tqdm
import json

def generate_insert_conversations(df, lesion_type, report_col, inserted_sentence_col, question_format, start_index=0):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['detection'])
        report = sample[report_col]
        distorted_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{distorted_sentence}"

        human_dict = {
            'type': 'insertion',
            'level': '2',
            'lesion': lesion_type,
            'from': 'human',
            'value': input_text.strip()
        }

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


full_list += generate_insert_conversations(none_insert_nodule, 
                                           lesion_type='nodule',
                                           report_col='insertion_nodule_distorted_report',
                                           inserted_sentence_col='insertion_nodule_inserted_sentence',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_insert_conversations(none_insert_effusion, 
                                           lesion_type='effusion',
                                           report_col='insertion_effusion_distorted_report',
                                           inserted_sentence_col='insertion_effusion_inserted_sentence',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_insert_conversations(insertion_effusion, 
                                           lesion_type='effusion',
                                           report_col='insertion_effusion_distorted_report',
                                           inserted_sentence_col='insertion_effusion_inserted_sentence',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_insert_conversations(insertion_nodule, 
                                           lesion_type='nodule',
                                           report_col='insertion_nodule_distorted_report',
                                           inserted_sentence_col='insertion_nodule_inserted_sentence',
                                           question_format=question_format,
                                           start_index=len(full_list))


### others

In [ ]:
import random
from tqdm import tqdm
import json

def generate_other_conversations(df, report_col, inserted_sentence_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['detection'])
        report = sample[report_col]
        distorted_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{distorted_sentence}"
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '2',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '2',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# nodule
full_list += generate_other_conversations(direction_nodule, 
                                          lesion_type='nodule',
                                          report_col='direction_nodule_distorted_report',
                                          inserted_sentence_col='direction_nodule_distorted_sentence',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_nodule, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_distorted_sentence',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_nodule, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_distorted_sentence',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_nodule, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_distorted_sentence',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))



# effusion
full_list += generate_other_conversations(direction_effusion, 
                                          lesion_type='effusion',
                                          report_col='direction_effusion_distorted_report',
                                          inserted_sentence_col='direction_effusion_distorted_sentence',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_effusion, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_distorted_sentence',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_effusion, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_distorted_sentence',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_effusion, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_distorted_sentence',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))


### No errors detected

In [ ]:
import random
from tqdm import tqdm
import json

def generate_normal_conversations(df, report_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['detection'])
        report = sample[report_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = "No errors detected."
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '2',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '2',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list



# normal
full_list += generate_normal_conversations(nodule_sample_df, 
                                           lesion_type='nodule',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_normal_conversations(effusion_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 

full_list += generate_normal_conversations(none_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 



In [ ]:
len(full_list)

## level 3

### omission

In [ ]:
import random
from tqdm import tqdm
import json

def generate_omit_conversations(df, lesion_type, report_col, inserted_sentence_col, question_format, start_index=0):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['correction'])
        report = sample[report_col]
        corrected_sentence = sample[inserted_sentence_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{corrected_sentence}"

        human_dict = {
            'type': 'omission',
            'level': '3',
            'lesion': lesion_type,
            'from': 'human',
            'value': input_text.strip()
        }

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# run
full_list += generate_omit_conversations(omission_nodule, 
                                         lesion_type='nodule',
                                         report_col='omission_nodule_distorted_report',
                                         inserted_sentence_col='omission_nodule_deleted_sentence',
                                         question_format=question_format,
                                         start_index=len(full_list))

full_list += generate_omit_conversations(omission_effusion, 
                                         lesion_type='effusion',
                                         report_col='omission_effusion_distorted_report',
                                         inserted_sentence_col='omission_effusion_deleted_sentence',
                                         question_format=question_format,
                                         start_index=len(full_list))



### others

In [ ]:
import random
from tqdm import tqdm
import json

def generate_other_conversations(df, report_col, inserted_sentence_col, corrected_sentence_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['correction'])
        report = sample[report_col]
        corrected_sentence = sample[corrected_sentence_col]
        
        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = f"{corrected_sentence}"
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '3',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '3',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# nodule
full_list += generate_other_conversations(direction_nodule, 
                                          lesion_type='nodule',
                                          report_col='direction_nodule_distorted_report',
                                          inserted_sentence_col='direction_nodule_distorted_sentence',
                                          corrected_sentence_col='direction_nodule_corrected_sentence',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_nodule, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_distorted_sentence',
                                          corrected_sentence_col='size_corrected_sentence',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_nodule, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_distorted_sentence',
                                          corrected_sentence_col='typo_corrected_sentence',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_nodule, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_distorted_sentence',
                                          corrected_sentence_col='unit_corrected_sentence',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))



# effusion
full_list += generate_other_conversations(direction_effusion, 
                                          lesion_type='effusion',
                                          report_col='direction_effusion_distorted_report',
                                          inserted_sentence_col='direction_effusion_distorted_sentence',
                                          corrected_sentence_col='direction_effusion_corrected_sentence',
                                          type='direction',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(size_effusion, 
                                          report_col='size_distorted_report',
                                          inserted_sentence_col='size_distorted_sentence',
                                          corrected_sentence_col='size_corrected_sentence',
                                          type='size',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(typo_effusion, 
                                          report_col='typo_distorted_report',
                                          inserted_sentence_col='typo_distorted_sentence',
                                          corrected_sentence_col='typo_corrected_sentence',
                                          type='typo',
                                          question_format=question_format,
                                          start_index=len(full_list))

full_list += generate_other_conversations(unit_effusion, 
                                          report_col='unit_distorted_report',
                                          inserted_sentence_col='unit_distorted_sentence',
                                          corrected_sentence_col='unit_corrected_sentence',
                                          type='unit',
                                          question_format=question_format,
                                          start_index=len(full_list))


### No errors detected

In [ ]:
import random
from tqdm import tqdm
import json

def generate_normal_conversations(df, report_col, question_format, type, lesion_type=None, start_index=len(full_list)):
    data_list = []
    for i in tqdm(range(len(df))):
        sample = df.iloc[i]
        cls_prompt = random.choice(question_format['correction'])
        report = sample[report_col]

        input_text = f"{cls_prompt}\nHere is the medical report: '''{report}'''"
        answer = "No errors detected."
        
        if lesion_type is not None:
            human_dict = {
                'type': type,
                'level': '3',
                'lesion': lesion_type,
                'from': 'human',
                'value': input_text.strip()
            }
        else:
            human_dict = {
                'type': type,
                'level': '3',
                'from': 'human',
                'value': input_text.strip()
            }            

        gpt_dict = {
            'from': 'gpt',
            'value': answer.strip()
        }

        conversation = [human_dict, gpt_dict]

        data_dict = {
            'id': f'report_generation_{start_index + i}',
            'image': sample['id'],
            'conversations': conversation
        }

        data_list.append(data_dict)
    
    return data_list


# normal
full_list += generate_normal_conversations(nodule_sample_df, 
                                           lesion_type='nodule',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list))

full_list += generate_normal_conversations(effusion_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 

full_list += generate_normal_conversations(none_sample_df, 
                                           lesion_type='effusion',
                                           report_col='lung_section',
                                           type='normal',
                                           question_format=question_format,
                                           start_index=len(full_list)) 



In [ ]:
len(full_list)

In [ ]:
import json

with open("./error_full_list_final_new.json", "w") as f:
    json.dump(full_list, f, indent=4)


In [ ]:
# split by task level

In [ ]:
level_1_list = []
level_2_list = []
level_3_list = []

for i in full_list:
    if i['conversations'][0]['level'] == "1":
        level_1_list.append(i)
    elif i['conversations'][0]['level'] == "2":
        level_2_list.append(i)
    elif i['conversations'][0]['level'] == "3":
        level_3_list.append(i)        

In [ ]:
for level, items in [("1", level_1_list), ("2", level_2_list), ("3", level_3_list)]:
    with open(f"./error_full_list_final_new_level{level}.json", "w") as f:
        json.dump(items, f, indent=4)